<h2>Are American Robins Really a Sign of Spring? An Analysis of Complete eBird Checklists in North Carolina from 2021-2025</h2>

<b>Question:</b> Are American Robins reported in a higher proportion of complete eBird checklists during spring than other times of the year in North Carolina?

In the United States, American Robins are known to be the first signs of spring, despite being considered a year-round bird. This project uses data from eBird to see if there is a meaningful increase in their reporting during spring.

<a href="https://ebird.org/home">eBird</a> is a platform that anyone can contribute to (citizen science).

In [11]:
import numpy as np
import matplotlib as plt
import pandas as pd

Data Description: The datasets are downloaded from <a href="https://science.ebird.org/en/use-ebird-data/download-ebird-data-products">here</a>. I filtered my data to be checklists with American Robins in North Carolina from 2021-2025.

The download comes with two datasets: Sampling Event Data and EBD (eBird Basic Dataset). 

In [ ]:
# load in data

# Sampling Event Data = contains EVERY checklist in NC from 2021-2025 regardless if American Robin was observed
df_all_checklists = pd.read_csv("data/ebd_US-NC_amerob_202101_202512_smp_relJun-2026_sampling.txt", 
                                sep="\t", 
                                usecols=["SAMPLING EVENT IDENTIFIER", "OBSERVATION DATE", "ALL SPECIES REPORTED"])

# EBD = eBird Basic Dataset, contains ONLY checklists in NC from 2021-2025 where American Robin was observed
df_ar_only_checklists = pd.read_csv("data/ebd_US-NC_amerob_202101_202512_smp_relJun-2026.txt", 
                                    sep="\t", 
                                    usecols=["SAMPLING EVENT IDENTIFIER"])

# keep only relevant columns
# Sampling Event Identifier: unique alphanumeric id for each "sampling event" ie checklist
# Observation Date: date the checklist was created
# All Species Reported: whether the checklist is complete or not. we want to filter to complete checklists only to reduce bias
# We only need the sampling event id column for EBD, because we will join these two. these are just all the rows that contain a robin, the other info is the same

In [56]:
print(df_all_checklists.head())
print(df_ar_only_checklists.head())

  OBSERVATION DATE SAMPLING EVENT IDENTIFIER  ALL SPECIES REPORTED
4       2021-01-01                 S78407716                     1
5       2021-01-01                S296450589                     1
6       2021-01-01                 S78450211                     1
7       2021-01-01                 S78359157                     1
9       2021-01-01                 S78393190                     1
  SAMPLING EVENT IDENTIFIER
0                 S78960242
1                 S79477418
2                 S79668245
3                 S79849364
4                 S79473462


In [ ]:
# Check and clean data
print((df_all_checklists.dtypes))

# check if there are any missing values
print(df_all_checklists["OBSERVATION DATE"].isna().sum())
print(df_all_checklists["SAMPLING EVENT IDENTIFIER"].isna().sum())
print(df_all_checklists["ALL SPECIES REPORTED"].isna().sum())

OBSERVATION DATE             object
SAMPLING EVENT IDENTIFIER    object
ALL SPECIES REPORTED          int64
dtype: object
0
0
0


In [59]:
# Convert to datetime
df_all_checklists["OBSERVATION DATE"] = pd.to_datetime(df_all_checklists["OBSERVATION DATE"])
print(df_all_checklists["OBSERVATION DATE"])

4         2021-01-01
5         2021-01-01
6         2021-01-01
7         2021-01-01
9         2021-01-01
             ...    
1416851   2025-12-31
1416852   2025-12-31
1416857   2025-12-31
1416859   2025-12-31
1416860   2025-12-31
Name: OBSERVATION DATE, Length: 1128849, dtype: datetime64[ns]


In [25]:
df_ar_only_checklists.head()

,SAMPLING EVENT IDENTIFIER
0,S78960242
1,S79477418
2,S79668245
3,S79849364
4,S79473462


In [24]:
# Check how "All Species Reported" is encoded
# 1 = True, completed checklist
# 0 = False, not completed checklist
# explain what a completed checklist means and its importance
df_all_checklists["ALL SPECIES REPORTED"].value_counts()

1    1128849
0     288013
Name: ALL SPECIES REPORTED, dtype: int64

In [40]:
# Filter complete checklists ONLY ie All Species Reported = 1
df_all_checklists = df_all_checklists.query("`ALL SPECIES REPORTED` == 1")

# Check that only complete checklists remain
print(len(df_all_checklists))
print(df_all_checklists["ALL SPECIES REPORTED"].sum())
# they are equal


1128849
1128849


In [63]:
# create a new variable: robin_reported for ebd
print(len(df_ar_only_checklists))

# Since every checklist contains a Robin observation, we can assign each row to have a new variable called robin_reported
# 1 = checklist contains robin
# 0 = checklist does not contain robin

df_ar_only_checklists["robin_reported"] = 1
print(df_ar_only_checklists.head())

396766
  SAMPLING EVENT IDENTIFIER  robin_reported
0                 S78960242               1
1                 S79477418               1
2                 S79668245               1
3                 S79849364               1
4                 S79473462               1


In [67]:
# merge
df_merged = pd.merge(left = df_all_checklists, right = df_ar_only_checklists, how = "left", on = "SAMPLING EVENT IDENTIFIER")
print(df_merged.head())

  OBSERVATION DATE SAMPLING EVENT IDENTIFIER  ALL SPECIES REPORTED  \
0       2021-01-01                 S78407716                     1   
1       2021-01-01                S296450589                     1   
2       2021-01-01                 S78450211                     1   
3       2021-01-01                 S78359157                     1   
4       2021-01-01                 S78393190                     1   

   robin_reported  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4             1.0  


In [71]:
# Now change all of the NaN to 0
df_merged["robin_reported"] = df_merged["robin_reported"].fillna(0)
print(df_merged.head())

  OBSERVATION DATE SAMPLING EVENT IDENTIFIER  ALL SPECIES REPORTED  \
0       2021-01-01                 S78407716                     1   
1       2021-01-01                S296450589                     1   
2       2021-01-01                 S78450211                     1   
3       2021-01-01                 S78359157                     1   
4       2021-01-01                 S78393190                     1   

   robin_reported  
0             0.0  
1             0.0  
2             0.0  
3             0.0  
4             1.0  


In [ ]:
# assign months and year
# Since we converted "OBSERVATION DATE" to datetime, we can use Series.dt.month
# maybe convert here instead of earlier?
# January = 1 ... December = 12
df_merged["year"] = df_merged["OBSERVATION DATE"].dt.year
df_merged["month"] = df_merged["OBSERVATION DATE"].dt.month
print(df_merged.head())

  OBSERVATION DATE SAMPLING EVENT IDENTIFIER  ALL SPECIES REPORTED  \
0       2021-01-01                 S78407716                     1   
1       2021-01-01                S296450589                     1   
2       2021-01-01                 S78450211                     1   
3       2021-01-01                 S78359157                     1   
4       2021-01-01                 S78393190                     1   

   robin_reported  month  year  
0             0.0      1  2021  
1             0.0      1  2021  
2             0.0      1  2021  
3             0.0      1  2021  
4             1.0      1  2021  


In [103]:
# now group by year and months
month_robins = df_merged.groupby(["year", "month"])["robin_reported", "ALL SPECIES REPORTED"].sum()
display(month_robins.head())

# we can treat All Species Reported as the total number of checklists in that month

/var/folders/5g/__stt9qj38lgdlp8f6vxlr000000gp/T/ipykernel_56682/486457396.py:2: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  month_robins = df_merged.groupby(["year", "month"])["robin_reported", "ALL SPECIES REPORTED"].sum()


robin_reported  ALL SPECIES REPORTED
year month                                      
2021 1              5393.0                 18267
     2              7059.0                 20806
     3              7920.0                 17129
     4              8060.0                 20607
     5              8490.0                 24119

In [102]:
# Rename ALL SPECIES REPORTED to total_checklists
month_robins = month_robins.rename(columns = {"ALL SPECIES REPORTED" : "total_checklists"})

# Reporting rate = robin_reported / total_checklists
month_robins["reporting_rate"] = month_robins["robin_reported"] / month_robins["total_checklists"]

# Convert to percentage and round to 2 decimal digits
month_robins["reporting_rate"] = (month_robins["reporting_rate"] * 100).round(2)

print(month_robins.head())

            robin_reported  total_checklists  reporting_rate
year month                                                  
2021 1              5393.0             18267           29.52
     2              7059.0             20806           33.93
     3              7920.0             17129           46.24
     4              8060.0             20607           39.11
     5              8490.0             24119           35.20
